# Executive Command Center Prototype

**Client/Stakeholder:** James Mitchell, Director of Sales Strategy  
**Developer:** Junior Data Analyst, Vizionary Analytics  
**Tech Lead:** Kwame Osei

## Objective
James needs a unified, interactive view of regional advertising campaigns and sales performance for tomorrow's Q3 planning meeting. This notebook implements the front-end Plotly visualizations for the command center: a two-panel core-metrics dashboard, a month-by-month animated scatter plot, and an interactive HTML export.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

### Data Loading
The historical extract `data/regional_sales_data.csv` is the primary data source. Its column is named `Ad_Spend`, so it is renamed to `Ad Spend` to match the dashboard labels. Months are stored as an ordered category so the charts and the animation run January → December rather than alphabetically.

Kwame's simulated live-data feed is kept as a fallback, so development can continue if the file is unavailable.

In [2]:
MONTH_ORDER = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

try:
    df = pd.read_csv('data/regional_sales_data.csv').rename(columns={'Ad_Spend': 'Ad Spend'})
    df['Month'] = pd.Categorical(df['Month'], categories=MONTH_ORDER, ordered=True)
    source = 'data/regional_sales_data.csv'
except FileNotFoundError:
    # Fallback: simulated live-data generation loop (mimics regional_sales_data.csv)
    np.random.seed(42)
    months = ['January', 'February', 'March', 'April', 'May', 'June']
    regions = ['North America', 'EMEA', 'APAC', 'LATAM']
    data = []
    for month in months:
        for region in regions:
            ad_spend = np.random.randint(15000, 75000)
            conversions = int((ad_spend / 100) * np.random.uniform(0.3, 1.8))
            revenue = conversions * np.random.uniform(150, 300)
            data.append([month, region, ad_spend, conversions, revenue])
    df = pd.DataFrame(data, columns=['Month', 'Region', 'Ad Spend', 'Conversions', 'Revenue'])
    df['Month'] = pd.Categorical(df['Month'], categories=months, ordered=True)
    MONTH_ORDER = months
    source = 'simulated feed (fallback)'

print(f'Data source: {source}  |  rows: {len(df)}  |  columns: {list(df.columns)}')
display(df.head())

Data source: data/regional_sales_data.csv  |  rows: 100  |  columns: ['Region', 'Month', 'Ad Spend', 'Revenue', 'Conversions']


,Region,Month,Ad Spend,Revenue,Conversions
0,North,Jan,12500,45200,320
1,South,Jan,15400,49000,410
2,East,Jan,9800,31000,215
3,West,Jan,22100,78000,640
4,North,Jan,11200,41500,290


## Task 1: Core Metrics Subplot Dashboard
James needs a side-by-side view to compare overall revenue across regions alongside raw campaign efficiency (Ad Spend vs. Conversions).

1. Initialize a 1x2 subplot figure.
2. Add a Bar chart to the first column.
3. Add a Scatter plot to the second column.

Both panels use the same colour for the same region, and hover tooltips show exact values.

In [3]:
# Step 1: Data Preparation for Subplot 1 (Aggregate Revenue by Region)
revenue_by_region = df.groupby('Region')['Revenue'].sum().reset_index()

# One consistent colour per region across both panels
region_colors = dict(zip(revenue_by_region['Region'], px.colors.qualitative.Set2))

# 1x2 subplot grid with the required subplot titles
dashboard_fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Revenue by Region', 'Ad Spend vs. Conversions')
)

# Panel 1: bar chart of Revenue (y) by Region (x)
dashboard_fig.add_trace(
    go.Bar(
        x=revenue_by_region['Region'],
        y=revenue_by_region['Revenue'],
        name='Revenue',
        marker_color=revenue_by_region['Region'].map(region_colors),
        hovertemplate='<b>%{x}</b><br>Total revenue: $%{y:,.0f}<extra></extra>'
    ),
    row=1, col=1
)

# Panel 2: scatter of Ad Spend (x) vs Conversions (y)
dashboard_fig.add_trace(
    go.Scatter(
        x=df['Ad Spend'],
        y=df['Conversions'],
        mode='markers',
        name='Campaigns',
        marker=dict(size=9, opacity=0.8, color=df['Region'].map(region_colors)),
        customdata=df[['Region', 'Month']],
        hovertemplate=('<b>%{customdata[0]}</b> (%{customdata[1]})<br>'
                       'Ad spend: $%{x:,.0f}<br>Conversions: %{y:,.0f}<extra></extra>')
    ),
    row=1, col=2
)

# Axis titles and number formats
dashboard_fig.update_xaxes(title_text='Region', row=1, col=1)
dashboard_fig.update_yaxes(title_text='Total revenue', tickprefix='$', tickformat=',', row=1, col=1)
dashboard_fig.update_xaxes(title_text='Ad spend', tickprefix='$', tickformat=',', row=1, col=2)
dashboard_fig.update_yaxes(title_text='Conversions', row=1, col=2)

# Finalizing layout
dashboard_fig.update_layout(
    height=500,
    title_text='Executive Command Center: Core Metrics',
    showlegend=False,
    template='plotly_white',
    hovermode='closest'
)
dashboard_fig.show()

## Task 2: Animated Scatter Plot
To help James understand how campaigns evolve over time, this animated chart progresses month-by-month. Bubble size shows conversions and colour shows region; the axes are fixed across frames so movement between months is easy to read.

In [4]:
animated_fig = px.scatter(
    df,
    x='Ad Spend',
    y='Revenue',
    animation_frame='Month',
    size='Conversions',
    color='Region',
    hover_name='Region',
    hover_data={'Ad Spend': ':$,.0f', 'Revenue': ':$,.0f', 'Conversions': ':,', 'Region': False, 'Month': False},
    category_orders={'Month': MONTH_ORDER},
    size_max=40,
    range_x=[0, df['Ad Spend'].max() * 1.1],
    range_y=[0, df['Revenue'].max() * 1.1],
    template='plotly_white'
)

animated_fig.update_layout(title_text='Month-by-Month Ad Spend vs Revenue Tracker')
animated_fig.show()

## Task 3: Export Deliverables
James will be presenting this from his tablet, so the core metrics dashboard is exported as a self-contained interactive HTML file (zoom, pan and hover work natively in the browser, with no internet connection required).

In [5]:
dashboard_fig.write_html('dashboard_export.html')

import os
print('Exported dashboard_export.html -', round(os.path.getsize('dashboard_export.html') / 1e6, 1), 'MB')

Exported dashboard_export.html - 4.8 MB
